In [9]:
from google import genai
from google.genai import types
client = genai.Client()


In [10]:
# from google import genai
# from google.genai import types

# # Define the function declaration for the model
# schedule_meeting_function = {
#     "name": "schedule_meeting",
#     "description": "Schedules a meeting with specified attendees at a given time and date.",
#     "parameters": {
#         "type": "object",
#         "properties": {
#             "attendees": {
#                 "type": "array",
#                 "items": {"type": "string"},
#                 "description": "List of people attending the meeting.",
#             },
#             "date": {
#                 "type": "string",
#                 "description": "Date of the meeting (e.g., '2024-07-29')",
#             },
#             "time": {
#                 "type": "string",
#                 "description": "Time of the meeting (e.g., '15:00')",
#             },
#             "topic": {
#                 "type": "string",
#                 "description": "The subject or topic of the meeting.",
#             },
#         },
#         "required": ["attendees", "date", "time", "topic"],
#     },
# }



# # Send request with function declarations
# response = client.models.generate_content(
#     model="gemini-3-flash-preview",
#     contents="Schedule a meeting with Bob and Alice for 03/14/2025 at 10:00 AM about the Q3 planning.",
#     config=config,
# )

# # Check for a function call
# if response.candidates[0].content.parts[0].function_call:
#     function_call = response.candidates[0].content.parts[0].function_call
#     print(f"Function to call: {function_call.name}")
#     print(f"ID: {function_call.id}")
#     print(f"Arguments: {function_call.args}")
#     #  In a real app, you would call your function here:
#     #  result = schedule_meeting(**function_call.args)
# else:
#     print("No function call found in the response.")
#     print(response.text)

In [11]:
client = genai.Client()

RUN_COMMAND = {
    "name": "run_command",
    "description": (
        "Run a shell command in the sandbox container. The sandbox has Python 3.13 and pip "
        "pre-installed. Use this to execute scripts, install packages, inspect files, "
        "or run any shell command. Commands run as bash -c so pipes and chaining work."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "command": {
                "type": "string",
                "description": (
                    "Shell command to execute as a clean string."
                ),
            },
        },
        "required": ["command"],
    },
}


user_part = types.Content(
    role="user",
    parts=[
        types.Part(text="what is the current time"),
        # types.Part.from_bytes(data=pdf_bytes, mime_type=pdf_mime)
    ],
)


contents = [user_part]


client.models.generate_content(
    model="gemini-3.1-pro-preview",
    contents=contents,
    config=types.GenerateContentConfig(
        temperature=1.0,  # "For Gemini 3, we strongly recommend keeping the temperature parameter at its default value of 1.0"
        max_output_tokens=65535,
        media_resolution=types.MediaResolution.MEDIA_RESOLUTION_MEDIUM,
        candidate_count=1,
        stop_sequences=[],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True  # https://discuss.ai.google.dev/t/model-gemini-2-5-pro-get-error-when-call-via-api/91119/2
        ),
        thinking_config=types.ThinkingConfig(
            thinking_level="low"
        ),
        tools=[types.Tool(function_declarations=[RUN_COMMAND])]
    ),
)

14:21:01 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              id=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b'\x12\xab\x03\n\xa8\x03\x01\xbe>\xf6\xfb\xd2\xfb\x029\xcb\x9c\xe5i\x16\x8f\x1bgS\x01>y\xd1\xf1h\xf1G\xba\xb5\x83\xb3\xf2O\x0c-\x8e\x18\x94e\xbd\xc9\x81HsfZ&\x18(F\x1b\x88\xe0\x19o?\x19\xb4\xbe\xc0\x0f`]{\x02P_\x9d\xca\xe6\xa7W\xb48\x16\xb5\xe8\x84\xfbA\xc06\xca\xcd\xaa\x05\x88Q|\x19*\xf3...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.1-pro-preview',
  response_id='jeLbac3GFJjc_uMPuuj3sAI',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=16,
    prompt_token_count=111,
   

In [12]:
import json
import numpy as np

def load_grid():
    """Load the current grid as a numpy array."""
    return np.array(json.load(open("state.json"))["grid"][-1])

def load_obs():
    """Load the full observation."""
    return json.load(open("state.json"))

def diff_grids(old, new):
    """Return list of (row, col, old_val, new_val) for changed cells."""
    return [(r, c, old[r][c], new[r][c])
            for r in range(len(old)) for c in range(len(old[r]))
            if old[r][c] != new[r][c]]

def color_counts(grid):
    """Count occurrences of each color value."""
    from collections import Counter
    return dict(Counter(int(v) for row in grid for v in row))

def find_color(grid, val):
    """Return list of (row, col) where grid == val."""
    return [(r, c) for r in range(len(grid)) for c in range(len(grid[r])) if grid[r][c] == val]

In [ ]:
"""Gemini agent that plays ARC-AGI-3 games."""

from arcengine import GameAction
from arc_agi import OperationMode
import arc_agi
from google.genai import types
from google import genai
from log import write_log, start_server, _render_grid
from typing import Any, Union
import argparse
import base64 as b64_mod
import hashlib
import json
import logging
import mimetypes
import subprocess
from dotenv import load_dotenv
load_dotenv()


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(message)s", datefmt="%H:%M:%S")


SYSTEM_PROMPT = """\
You are playing an interactive puzzle game on a 64x64 grid.

Each cell is an integer 0-15 representing a color. You don't know the game rules. \
You must figure them out by experimenting: take actions, observe what changes, \
build a theory, test it.

After each action you receive:
- state: NOT_FINISHED, WIN, or GAME_OVER
- score: levels completed so far (a score increase means a level was solved)
- win_levels: total levels needed to win
- available_actions: which actions are legal right now (only use these)
- A board image showing the current grid state
- A diff of what changed since the last action
- A memory of actions already tried from this board state

The full 64x64 grid is also written to /home/agent/state.json after every action.

Actions:
- ACTION1: up
- ACTION2: down
- ACTION3: left
- ACTION4: right
- ACTION5: interact / confirm (game-specific)
- ACTION6: click a cell at (x, y) where x,y in [0,63]
- ACTION7: undo
- RESET: restart the current level

Color palette (0-15):
0=white 1=off-white 2=light-gray 3=gray 4=off-black 5=black \
6=magenta 7=light-magenta 8=red 9=blue 10=light-blue 11=yellow \
12=orange 13=maroon 14=green 15=purple

You have a Linux sandbox (working dir /home/agent) with Python 3.12, numpy, \
matplotlib, pillow, and sudo. You can run any command, write scripts, install packages.

A helpers.py file is pre-loaded with:
- load_grid() → numpy array of the current 64x64 grid
- load_obs() → full observation dict
- diff_grids(old, new) → list of (row, col, old_val, new_val)
- color_counts(grid) → {color: count}
- find_color(grid, val) → [(row, col), ...]
Use: python3 -c "from helpers import *; grid = load_grid(); print(color_counts(grid))"

Goal: reach WIN state in as few actions as possible.\
"""

RUN_COMMAND = {
    "name": "run_command",
    "description": (
        "Run a shell command in the sandbox. Working directory is /home/agent. "
        "Python 3.12, numpy, matplotlib, pillow are available. "
        "Commands run as bash -c so pipes and chaining work."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "command": {
                "type": "string",
                "description": "Shell command to execute as a clean string.",
            },
        },
        "required": ["command"],
    },
}

VIEW_FILE = {
    "name": "view_file",
    "description": (
        "View an image or PDF from the sandbox. The file is added to your visual "
        "context so you can see and analyze it. Supports images (png, jpg, gif, webp) "
        'and PDFs. Do NOT use this for text files — read those with run_command("cat path") instead.'
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "Absolute path to an image or PDF in the sandbox.",
            },
        },
        "required": ["path"],
    },
}

TAKE_ACTION = {
    "name": "take_action",
    "description": (
        "Submit a game action. Returns state, score, available_actions, a board image, "
        "a diff of changes, and memory of previously tried actions. "
        "For ACTION6 (click), x and y are required (0-63)."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "action": {
                "type": "string",
                "enum": ["ACTION1", "ACTION2", "ACTION3", "ACTION4",
                         "ACTION5", "ACTION6", "ACTION7", "RESET"],
                "description": "The action to take.",
            },
            "x": {
                "type": "integer",
                "description": "X coordinate for ACTION6 (0-63). Ignored for other actions.",
            },
            "y": {
                "type": "integer",
                "description": "Y coordinate for ACTION6 (0-63). Ignored for other actions.",
            },
        },
        "required": ["action"],
    },
}

RENDER_BOARD = {
    "name": "render_board",
    "description": "Render the current game board as a PNG image. No arguments needed.",
    "parameters": {
        "type": "object",
        "properties": {},
    },
}

TOOLS = [RUN_COMMAND, VIEW_FILE, TAKE_ACTION, RENDER_BOARD]


# --- Game state ---

env = None
current_obs = None
state_action_memory = {}  # {grid_hash: {action: {"changed": bool, "diff": str}}}


def to_dict(obs):
    return {
        "state": obs.state.name,
        "score": obs.levels_completed,
        "win_levels": obs.win_levels,
        "grid": [layer.tolist() for layer in obs.frame],
        "available_actions": [GameAction.from_id(a).name for a in obs.available_actions],
    }


def _hash_grid(grid):
    return hashlib.md5(str(grid).encode()).hexdigest()[:12]


def _diff_grids(old, new):
    """Compact diff of changed cells."""
    if not old or not new:
        return ""
    old_g = old[-1] if isinstance(old[0],
                                  list) and isinstance(old[0][0], list) else old
    new_g = new[-1] if isinstance(new[0],
                                  list) and isinstance(new[0][0], list) else new
    changes = []
    for r in range(min(len(old_g), len(new_g))):
        for c in range(min(len(old_g[r]), len(new_g[r]))):
            if old_g[r][c] != new_g[r][c]:
                changes.append(f"{old_g[r][c]}→{new_g[r][c]} at ({r},{c})")
    if not changes:
        return "no change"
    if len(changes) > 20:
        return f"{len(changes)} cells changed. first 20: " + "; ".join(changes[:20])
    return "; ".join(changes)


def _get_tried_actions(grid):
    h = _hash_grid(grid)
    tried = state_action_memory.get(h, {})
    if not tried:
        return ""
    lines = []
    for action, result in tried.items():
        status = "changed" if result["changed"] else "no change"
        line = f"  {action}: {status}"
        if result["changed"] and result["diff"]:
            line += f" ({result['diff'][:100]})"
        lines.append(line)
    return "Already tried from this board state:\n" + "\n".join(lines)


def _record_action(grid, action_name, changed, diff):
    h = _hash_grid(grid)
    state_action_memory.setdefault(h, {})[action_name] = {
        "changed": changed, "diff": diff}


def _sync_state_to_sandbox(obs):
    state_json = json.dumps(obs)
    subprocess.run(
        ["docker", "exec", "-w", "/home/agent", "sandbox",
         "bash", "-c", f"cat > state.json << 'STATEEOF'\n{state_json}\nSTATEEOF"],
        capture_output=True, timeout=10,
    )


def _seed_sandbox_helpers():
    helpers = '''\
import json
import numpy as np

def load_grid():
    """Load the current grid as a numpy array."""
    return np.array(json.load(open("state.json"))["grid"][-1])

def load_obs():
    """Load the full observation."""
    return json.load(open("state.json"))

def diff_grids(old, new):
    """Return list of (row, col, old_val, new_val) for changed cells."""
    return [(r, c, old[r][c], new[r][c])
            for r in range(len(old)) for c in range(len(old[r]))
            if old[r][c] != new[r][c]]

def color_counts(grid):
    """Count occurrences of each color value."""
    from collections import Counter
    return dict(Counter(int(v) for row in grid for v in row))

def find_color(grid, val):
    """Return list of (row, col) where grid == val."""
    return [(r, c) for r in range(len(grid)) for c in range(len(grid[r])) if grid[r][c] == val]
'''
    subprocess.run(
        ["docker", "exec", "-w", "/home/agent", "sandbox",
         "bash", "-c", f"cat > helpers.py << 'HELPEOF'\n{helpers}\nHELPEOF"],
        capture_output=True, timeout=10,
    )


def _render_board_bytes():
    """Render current board as PNG bytes, or None."""
    if not current_obs:
        return None
    b64 = _render_grid(current_obs["grid"])
    return b64_mod.b64decode(b64) if b64 else None


def step(args):
    global current_obs
    action_name = args["action"]

    if action_name == "RESET":
        obs = to_dict(env.reset())
        current_obs = obs
        _sync_state_to_sandbox(obs)
        return obs

    ga = GameAction.from_name(action_name)
    data = {"x": int(args["x"]), "y": int(args["y"])
            } if action_name == "ACTION6" else None
    obs = to_dict(env.step(ga, data=data))
    current_obs = obs
    _sync_state_to_sandbox(obs)
    return obs


# --- Validation ---

VALID_ACTIONS = {"ACTION1", "ACTION2", "ACTION3", "ACTION4",
                 "ACTION5", "ACTION6", "ACTION7", "RESET"}


def validate_action(args) -> bool:
    """
    Return False if the action args pass validation otherwise make error message
    """
    action = args.get("action")
    if action not in VALID_ACTIONS:
        return f"invalid action: {action}"
    if action != "RESET" and current_obs and action not in current_obs.get("available_actions", []):
        return f"action {action} not available. available: {current_obs['available_actions']}"
    if action == "ACTION6":
        x, y = args.get("x"), args.get("y")
        if x is None or y is None:
            return "ACTION6 requires x and y coordinates"
        if not (0 <= x <= 63 and 0 <= y <= 63):
            return f"coordinates out of range: x={x}, y={y} (must be 0-63)"
    return False


# --- Tool execution ---

def execute_tool(name, args) -> dict[str, Any]:
    try:
        return _execute_tool(name, args)
    except subprocess.TimeoutExpired:
        return {"result": "error: command timed out"}
    except Exception as e:
        return {"result": f"error: {type(e).__name__}: {e}"}


def _execute_tool(name, args) -> dict[str, Any]:
    if name == "run_command":
        result = subprocess.run(
            ["docker", "exec", "-w", "/home/agent",
                "sandbox", "bash", "-c", args["command"]],
            capture_output=True, text=True, timeout=60,
        )
        return {"result": (result.stdout + result.stderr).strip()}

    if name == "view_file":
        path = args["path"]
        result = subprocess.run(
            ["docker", "exec", "sandbox", "cat", path],
            capture_output=True, timeout=30,
        )
        if result.returncode != 0:
            return {"result": f"error: {result.stderr.decode().strip()}"}
        mime = mimetypes.guess_type(path)[0] or "application/octet-stream"
        return {"result": f"Displaying {path}", "_bytes": result.stdout, "_mime": mime}

    if name == "take_action":
        error = validate_action(args)
        if error:
            return {"result": f"error: {error}"}
        before_grid = current_obs["grid"] if current_obs else None
        obs = step(args)
        summary = {k: v for k, v in obs.items() if k != "grid"}

        action_name = args["action"]
        changed = before_grid is not None and obs["grid"] != before_grid
        diff = _diff_grids(before_grid, obs["grid"]) if before_grid else ""
        if before_grid:
            _record_action(before_grid, action_name, changed, diff)
        if not changed and before_grid is not None:
            summary["note"] = "grid unchanged — no effect."
        elif diff and diff != "no change":
            summary["diff"] = diff

        tried = _get_tried_actions(obs["grid"])
        if tried:
            summary["memory"] = tried

        img = _render_board_bytes()
        if img:
            return {"result": summary, "_bytes": img, "_mime": "image/png"}
        return {"result": summary}

    if name == "render_board":
        img = _render_board_bytes()
        if not img:
            return {"result": "error: no game state yet"}
        return {
            "result": f"Board rendered (state={current_obs['state']} score={current_obs['score']}/{current_obs['win_levels']})",
            "_bytes": img,
            "_mime": "image/png",
        }

    return {"result": f"unknown tool: {name}"}


MODELS = ["gemini-3-flash-preview", "gemini-3.1-pro-preview"]

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=[types.Tool(function_declarations=TOOLS)],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True),
    temperature=1.0,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_MEDIUM,
    candidate_count=1,
    stop_sequences=[],
    thinking_config=types.ThinkingConfig(thinking_level="high"),
)


class JackGamer():
    def __init__(self):
        self.client = genai.Client()
        self.model = MODELS[1]
        self.contents: list[types.Content] = []
        self.clear()

    def clear(self):
        prompt = (
            f"New game started. Initial state:\n{json.dumps(obs_summary, indent=2)}\n\n"
            f"The full grid is in /home/agent/state.json. "
            f"Start by calling render_board to see the board, then take actions to explore."
        )
        self.contents = [types.Content(
            role="user", parts=[types.Part(text=prompt)])]

    def generate_response(self) -> types.GenerateContentResponse:
        res = self.client.models.generate_content(
            model=self.model, contents=self.contents, config=CONFIG,
        )
        model_content = res.candidates[0].content
        calls = [p for p in model_content.parts if p.function_call]

        if not calls:
            print("did not make a tool call")
            self.contents.append(model_content)

        result_parts = []

        for p in calls:
            print(f"made {len(calls)} tool calls")
            fc: types.FunctionCall = p.function_call
            output: dict = execute_tool(fc.name, fc.args)

            if fc.name == "take_action":
                action_count += 1
                print("[%d] %s  state=%s  score=%d/%d", action_count, fc.args.get("action"),
                      current_obs["state"], current_obs["score"], current_obs["win_levels"])
            else:
                preview = output["result"] if isinstance(
                    output["result"], str) else str(output["result"])
                print("%s: %s", fc.name, preview[:200])

            fr_kwargs = {
                "name": fc.name,
                "response": {"result": output["result"]},
                "id": fc.id,
            }
            if "_bytes" in output:
                fr_kwargs["parts"] = [types.FunctionResponsePart(
                    inline_data=types.FunctionResponseBlob(
                        mime_type=output["_mime"],
                        display_name="board.png",
                        data=output["_bytes"],
                    )
                )]

            result_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(**fr_kwargs))
            )

        self.contents.append(types.Content(role="user", parts=result_parts))


# --- Agent loop ---
max_actions = 500
mode = "normal"
game_id = 'ls20'

env, current_obs = None, None

r = subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", "sandbox"],
                   capture_output=True, text=True)
if r.returncode != 0 or "true" not in r.stdout.lower():
    raise RuntimeError(
        "sandbox container not running. run: docker compose up -d")

arcade = arc_agi.Arcade(operation_mode=OperationMode(mode))
card_id = arcade.open_scorecard(tags=["agent"])
env = arcade.make(game_id, scorecard_id=card_id)

start_server()

current_obs = to_dict(env.reset())
_sync_state_to_sandbox(current_obs)
_seed_sandbox_helpers()
state_action_memory.clear()
logger.info("game=%s  state=%s  score=%d/%d", game_id,
            current_obs["state"], current_obs["score"], current_obs["win_levels"])

obs_summary = {k: v for k, v in current_obs.items() if k != "grid"}
# prompt = (
#     f"New game started. Initial state:\n{json.dumps(obs_summary, indent=2)}\n\n"
#     f"The full grid is in /home/agent/state.json. "
#     f"Start by calling render_board to see the board, then take actions to explore."
# )
# contents = [types.Content(role="user", parts=[types.Part(text=prompt)])]

jackagent = JackGamer()


while True:

    m = response.usage_metadata
    if m:
        usage["prompt_tokens"] = m.prompt_token_count or 0
        usage["prompt_tokens_total"] += m.prompt_token_count or 0
        usage["output_tokens"] += m.candidates_token_count or 0
        usage["thinking_tokens"] += m.thoughts_token_count or 0

    model_content = response.candidates[0].content
    calls = [p for p in model_content.parts if p.function_call]

    if not calls:
        text = (response.text or "").strip()
        logger.info("model: %s", text[:200] if text else "(thinking)")
        contents.append(model_content)
        write_log(contents, usage=usage, model=MODEL)
        continue


scorecard = arcade.close_scorecard(card_id)
if scorecard:
    logger.info("scorecard: %s", card_id)
    logger.info(json.dumps(scorecard.model_dump(), indent=2, default=str))

2026-04-12 14:21:01 | INFO | Successfully fetched 25 environment(s) from API


14:21:01 Successfully fetched 25 environment(s) from API


2026-04-12 14:21:01 | INFO | Created new scorecard: 7aea5578-062f-4c71-b1cb-5e6b7af7ef6d


14:21:01 Created new scorecard: 7aea5578-062f-4c71-b1cb-5e6b7af7ef6d


2026-04-12 14:21:01 | INFO | Successfully fetched metadata for game ls20


...

2026-04-12 14:21:02 | INFO | Successfully downloaded game ls20 (version: 9607627b) to environment_files/ls20/9607627b


14:21:02 Successfully downloaded game ls20 (version: 9607627b) to environment_files/ls20/9607627b


2026-04-12 14:21:02 | INFO | Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py


14:21:02 Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py
14:21:02 game=ls20  state=NOT_FINISHED  score=0/7
14:21:02 calling gemini (0 actions so far)...
14:21:05 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
14:21:05 run_command: {5: 439, 4: 2609, 3: 892, 9: 45, 0: 3, 1: 2, 12: 10, 11: 84, 8: 12}
14:21:05 render_board: Board rendered (state=NOT_FINISHED score=0/7)
14:21:05 calling gemini (0 actions so far)...
14:21:10 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
14:21:10 run_command: white 0 [(31, 21), (32, 21), (32, 22)]
blue 9 [(11, 35), (11, 36), (11, 37), (12, 37), (13, 35), (13, 37), (47, 34), (47, 35), (47, 36), (47, 37), (47, 38), (48, 34), (48, 35), (48, 36), (48, 37), (48
14:21:10 calling gemini (0 actions so far)...
14:21:13 HTTP Request: POST https://generativelanguage.goo

KeyboardInterrupt: 

In [14]:
contents

[Content(
   parts=[
     Part(
       text="""New game started. Initial state:
 {
   "state": "NOT_FINISHED",
   "score": 0,
   "win_levels": 7,
   "available_actions": [
     "ACTION1",
     "ACTION2",
     "ACTION3",
     "ACTION4"
   ]
 }
 
 The full grid is in /home/agent/state.json. Start by calling render_board to see the board, then take actions to explore."""
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'command': 'python3 -c "from helpers import *; grid = load_grid(); print(color_counts(grid))"'
         },
         id='lnx2sgw4',
         name='run_command'
       ),
       thought_signature=b'\x12\x8f\x03\n\x8c\x03\x01\xbe>\xf6\xfb\xefE\xeapw\x88\xe0\xbeE\'\xc3\xba \xb8\x9a9\x9b\x99\xc1\xf4\xfa\x10\x14\xc0y\xd3\xf5\x01<\xfc\x02\xef\xc0\x81\x86V\xb2\xf4!DN`;\x1d!\x91\x14R\xc0\xf7\x1313K\x1b<\x8b\xa8/\x1e\x839"\x8bC\xf8Y\xa2\x81.\\\xd5n\x95\xe3\xc8\xd4\x9fQ\x82x4k\x1d\x9b\xaf...'
     ),
     Pa

In [15]:
state_action_memory

{'06ff5221f0f4': {'ACTION4': {'changed': True,
   'diff': '52 cells changed. first 20: 12→3 at (45,34); 12→3 at (45,35); 12→3 at (45,36); 12→3 at (45,37); 12→3 at (45,38); 3→12 at (45,39); 3→12 at (45,40); 3→12 at (45,41); 3→12 at (45,42); 3→12 at (45,43); 12→3 at (46,34); 12→3 at (46,35); 12→3 at (46,36); 12→3 at (46,37); 12→3 at (46,38); 3→12 at (46,39); 3→12 at (46,40); 3→12 at (46,41); 3→12 at (46,42); 3→12 at (46,43)'}},
 '9a99f6fe685d': {'ACTION1': {'changed': True,
   'diff': '52 cells changed. first 20: 3→12 at (40,39); 3→12 at (40,40); 3→12 at (40,41); 3→12 at (40,42); 3→12 at (40,43); 3→12 at (41,39); 3→12 at (41,40); 3→12 at (41,41); 3→12 at (41,42); 3→12 at (41,43); 3→9 at (42,39); 3→9 at (42,40); 3→9 at (42,41); 3→9 at (42,42); 3→9 at (42,43); 3→9 at (43,39); 3→9 at (43,40); 3→9 at (43,41); 3→9 at (43,42); 3→9 at (43,43)'}},
 'aa1286321dde': {'ACTION1': {'changed': True,
   'diff': '52 cells changed. first 20: 3→12 at (35,39); 3→12 at (35,40); 3→12 at (35,41); 3→12 at (35,